# LSTM-arithmetic



In [ ]:
# ! pip install seaborn
# ! pip install opencc
# ! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
# Generative AI Usage: Used OpenAI Codex as a coding assistant under my direction. I defined the requirements, selected the approaches, reviewed and tested the generated code, and iteratively refined the implementation.
# Install the template's optional OpenCC dependency only when it is missing.
try:
    import opencc
except ModuleNotFoundError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'opencc-python-reimplemented'])
    import opencc
import os
from sklearn.model_selection import train_test_split

# VS Code Colab uses remote files: upload data/ or put HW2/data on Drive.
# Optional overrides: HW2_DATA_DIR (CSVs), HW2_OUTPUT_DIR (persistent artifacts).
from pathlib import Path
import sys
import platform
import importlib.metadata

candidates = [Path(os.environ['HW2_DATA_DIR'])] if 'HW2_DATA_DIR' in os.environ else [
    Path('./data'), Path('./HW2/data'), Path('/content/data'), Path('/content'),
    Path('/content/drive/MyDrive/HW2/data'),
]
if not any(all((p / name).is_file() for name in ['arithmetic_train.csv', 'arithmetic_eval.csv'])
           for p in candidates):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass
    candidates.append(Path('/content/drive/MyDrive/HW2/data'))
data_path = next((p for p in candidates if all((p / name).is_file() for name in
                 ['arithmetic_train.csv', 'arithmetic_eval.csv'])), None)
if data_path is None:
    raise FileNotFoundError('Upload both CSVs to data/ or set HW2_DATA_DIR to their remote directory.')
output_path = Path(os.environ.get('HW2_OUTPUT_DIR', str(data_path.parent if data_path.name == 'data' else data_path)))
output_path.mkdir(parents=True, exist_ok=True)
print('Data:', data_path.resolve(), '| Artifacts:', output_path.resolve())
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv'))
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv'))
df_train.head()

In [ ]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x))
df_train['src'] = df_train['src'].add(df_train['tgt'])
df_train['len'] = df_train['src'].apply(lambda x: len(x))

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x))

# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [ ]:
char_to_id = {}
id_to_char = {}

# Generative AI Usage: Used OpenAI Codex as a coding assistant under my direction. I defined the requirements, selected the approaches, reviewed and tested the generated code, and iteratively refined the implementation.
# Stable IDs, constructed from training characters only (no eval leakage).
characters = sorted(set(''.join(df_train['src'])))
char_to_id = {token: index for index, token in enumerate(['<pad>', '<eos>'] + characters)}
id_to_char = {index: token for token, index in char_to_id.items()}
# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id converts characters to IDs; id_to_char performs the reverse mapping.

vocab_size = len(char_to_id)
print('Vocab size{}'.format(vocab_size))

# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [ ]:
# Teacher forcing: x contains the prompt and true answer, without EOS.
# At '=' predict the first answer character; at the final digit predict EOS.
pad_id = char_to_id['<pad>']
eos_id = char_to_id['<eos>']
def encode_training_row(sequence):
    answer_start = sequence.index('=') + 1
    x = [char_to_id[c] for c in sequence]
    y = [pad_id] * (answer_start - 1) + x[answer_start:] + [eos_id]
    return x, y
encoded = [encode_training_row(s) for s in df_train['src']]
df_train['char_id_list'] = [item[0] for item in encoded]
df_train['label_id_list'] = [item[1] for item in encoded]
del encoded
max_answer_len = int(df_train['tgt'].str.len().max())
df_train.head()

# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of examples in a batch|128|
|`epochs`|Epochs for reproducing the selected checkpoint; the recorded run includes 28 epochs|26|
|`embed_dim`|Character embedding dimension|128|
|`hidden_dim`|Hidden-state dimension of each LSTM|512|
|`lr`|Initial learning rate|0.001|
|`grad_clip`|Elementwise gradient clipping bound|1|


In [ ]:
batch_size = 128
epochs = 26
embed_dim = 128
hidden_dim = 512
lr = 0.001
grad_clip = 1


# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- Then, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    
    def __len__(self):
        # return the amount of data
        return len(self.sequences)
    
    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        row = self.sequences.iloc[index]
        x = row['char_id_list']
        y = row['label_id_list']
        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])
    
    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])
    
    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [ ]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [ ]:
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(
    ds_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn,
    num_workers=0, pin_memory=torch.cuda.is_available(),
    generator=torch.Generator().manual_seed(2))

# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [ ]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()
        # Write your code here
        # Design your own model, which should contain embedding layer, several hidden layers and a linear layer
        
        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                            embedding_dim=embed_dim,
                                            padding_idx=char_to_id['<pad>'])
        
        self.rnn_layer1 = torch.nn.LSTM(input_size=embed_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.rnn_layer2 = torch.nn.LSTM(input_size=hidden_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)
        
        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=hidden_dim),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=vocab_size))
        
    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)
    
    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)
        
        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)
        
        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)
        
        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)
        
        batch_x = self.linear(batch_x)
        
        return batch_x
    
    def generator(self, start_char, max_len=200):
        
        char_list = [char_to_id[c] for c in start_char]
        
        next_char = None
        
        while len(char_list) < max_len: 
            # Cache the two LSTM states: consume the prompt once, then only
            # the previous generated character. No true answer is supplied.
            with torch.no_grad():
                first_step = len(char_list) == len(start_char)
                token_ids = char_list if first_step else [char_list[-1]]
                x = torch.tensor([token_ids], dtype=torch.long,
                                 device=next(self.parameters()).device)
                x = self.embedding(x)
                x, state1 = self.rnn_layer1(x, None if first_step else state1)
                x, state2 = self.rnn_layer2(x, None if first_step else state2)
                y = self.linear(x[:, -1, :])
                next_char = int(y.argmax(dim=-1).item())
            
            if next_char == char_to_id['<eos>']:
                break
            
            char_list.append(next_char)
            
        return [id_to_char[ch_id] for ch_id in char_list]

In [ ]:
torch.manual_seed(2)


device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
if device.type == 'cpu':
    raise RuntimeError('HW2 requires GPU training. Select a Colab GPU runtime or an available Apple GPU.')
if device.type == 'cuda':
    torch.cuda.manual_seed_all(2)
print('Running environment:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'Apple GPU (MPS)')
print('Python version:', platform.python_version())

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim)

In [ ]:
model = model.to(device)
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=1, threshold=0.001, min_lr=1e-5)
best_em = -1.0
history = []
# Record versions from the actual training kernel, rather than guessed pins.
package_names = ['numpy', 'pandas', 'torch', 'matplotlib', 'seaborn',
                 'scikit-learn', 'tqdm']
# Different distributions can provide the template's opencc import.
package_names += importlib.metadata.packages_distributions().get('opencc', [])
(output_path / 'requirements.txt').write_text(''.join(
    f'{name}=={importlib.metadata.version(name)}\n' for name in package_names), encoding='utf-8')

# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `50` batches, output the current loss to monitor whether it is converging.

In [ ]:
from tqdm import tqdm
from copy import deepcopy
model = model.to(device)
model.train()
i = 0
for epoch in range(1, epochs+1):
    # The process bar
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")
    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        # Generative AI Usage: Used OpenAI Codex as a coding assistant under my direction. I defined the requirements, selected the approaches, reviewed and tested the generated code, and iteratively refined the implementation.
        model.train()
        optimizer.zero_grad(set_to_none=True)
    
        batch_pred_y = model(batch_x.to(device), batch_x_lens)
        
        loss = criterion(batch_pred_y.reshape(-1, vocab_size),
                         batch_y.to(device).reshape(-1))
        loss.backward()

        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip) # gradient clipping

        optimizer.step()

        i+=1
        if i%50==0:
            bar.set_postfix(loss = loss.item())
    
    # Evaluate your model
    matched = 0
    total = 0
    bar_eval = tqdm(df_eval.iterrows(), desc=f"Validation epoch {epoch}")
    for _, row in bar_eval:
        batch_x = row['src']
        batch_y = row['tgt']
        
        model.eval()
        prediction = ''.join(model.generator(batch_x, max_len=len(batch_x) + max_answer_len + 1))
        prediction = prediction[len(batch_x):]
        
        matched += int(prediction == batch_y)
        total += 1

    em = matched / total
    history.append({'epoch': epoch, 'exact_match': em, 'lr': optimizer.param_groups[0]['lr']})
    if em > best_em:
        best_em = em
        # Plain state_dict: reconstruct CharRNN and load with map_location.
        torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()},
                   output_path / 'checkpoint.pt')
        print(f'Best checkpoint saved: epoch={epoch}, EM={em:.6f}')
    scheduler.step(em)
    print('EM history (copy into report):', history)
    print('Checkpoint:', output_path / 'checkpoint.pt')

        
    print(matched/total)

I trained the model with an experiment launcher on Apple M5 MPS.
Selected checkpoint: epoch 26; recorded full-evaluation EM: 0.988500.
I independently verified the saved checkpoint on CPU using this notebook's generator: 9885/10000 = 0.988500.
These results summarize the recorded experiment and independent verification; this notebook was not rerun end to end.


Running environment: I trained the main model on an Apple M5 GPU using MPS and independently verified the saved checkpoint on CPU. I ran the earlier baseline on a Colab Tesla T4.

Python version: 3.13.15; PyTorch 2.15.0. The accompanying requirements.txt records the package versions of the selected experiment.

##### 1. Describe your model architecture, data preprocessing methods, and hyperparameter settings, and justify your choices.  (5%)

Answer: I used a character-level model: Embedding(128) → LSTM(512) → LSTM(512) → Linear(512,512) → ReLU → Linear(512,18), without attention. I built the vocabulary from the training set only, including `<pad>` and `<eos>`. For teacher forcing, I masked the labels before the equals sign and the batch padding with `<pad>`, which the cross-entropy loss ignores. At the equals sign, the model predicts the first answer character; at the last answer character, it predicts EOS. During generation, I supplied only the expression, without the true answer.

My settings were batch_size=128, embed_dim=128, hidden_dim=512, Adam with an initial learning rate of 0.001, and clip_grad_value_=1. I used ReduceLROnPlateau to monitor full-answer generation exact match (EM), with factor=0.5, patience=1, threshold=0.001, and min_lr=1e-5. I chose a larger hidden state to represent operands and parentheses, and a smaller embedding to reduce computation. These settings were selected using the evaluation set.

I generated answers in the usual decimal order. The best checkpoint was from epoch 26, with full-answer EM of 98.85%. After loading the saved weights, I independently generated answers for all 10,000 evaluation expressions on CPU using this notebook's generator and reproduced 98.85% EM (9885/10000).

##### 2. If you use 2-layer LSTM or GRU instead of RNN, what will happen to the quality of your answer generation? Why? (10%)

Answer: In my controlled experiment, both the two-layer LSTM and the two-layer GRU achieved higher generation EM than the two-layer vanilla RNN. I used the same 50,000 training examples, 1,000 evaluation examples, seed=2, embedding=64, hidden=64, batch=128, and 10 training epochs. All models used Adam with lr=0.001, elementwise gradient clipping at 1, and no scheduler. I report the best full-answer EM within the same epoch budget. This comparison fixes the dimensions rather than the parameter count; its scores are not directly comparable to those of the main model trained on one million examples and evaluated on 10,000 examples.

| Model | Best generation EM | Parameters |
|---|---:|---:|
| RNN | 6.50% | 23122 |
| LSTM | 9.00% | 73042 |
| GRU | 10.60% | 56402 |

The observed ordering was GRU > LSTM > RNN under this configuration. LSTM's cell state and input, forget, and output gates, and GRU's update and reset gates, provide mechanisms for retaining information across characters and mitigating vanishing gradients. GRU also had fewer parameters than LSTM at the same hidden dimension. These mechanisms offer an explanation for the observed advantage, but I did not isolate them in a separate causal experiment. The small models may still be undertrained, and I used only one seed, so the result does not establish a universal ordering or demonstrate mastery of an exact arithmetic algorithm.

##### 3. If we construct an training set using three-digit numbers while the evaluation set is constructed from two-digit numbers, what will happen to the quality of your answer generation? (10%)

Answer: In theory, training on three-digit numbers and evaluating on two-digit numbers introduces a distribution shift in input length, digit positions, answer range, and carry patterns. Sharing the same character vocabulary does not imply that the model has learned an arithmetic algorithm that transfers across digit lengths. Generation EM may decrease, and shorter two-digit inputs alone do not guarantee easier generalization.

I completed a separate two-digit pilot using 50,000 training expressions with operands from 10–99 and 1,000 two-digit evaluation expressions whose source strings did not overlap with the training set. A two-layer LSTM with embedding/hidden=64/64, batch=128, seed=2, Adam lr=0.001, and 10 epochs achieved a best full-answer EM of 2.70%. This small model was already insufficient for reliable generation within the same digit range.

I do not have a completed, comparable three-digit training experiment with operands from 100–999, so I cannot report a measured three-digit-to-two-digit transfer effect. The distribution-shift discussion above remains a theoretical analysis. I used the original one-million-example dataset unchanged for the main model and did not mix these synthetic training examples into it.

##### 4. If we construct a training set that includes 20% incorrect answers, how will this affect the quality of the generated responses? Present some examples. (10%)

Answer: In my experiment, corrupting 20% of the training answers reduced the best clean-evaluation generation EM from 9.00% to 7.80%, a decrease of 1.20 percentage points. With a fixed seed, I randomly selected exactly 10,000 of the same 50,000 training examples and changed the final answer digit. I preserved the expression, answer length, and minus sign, and ensured that each corrupted answer differed from the correct answer. This experiment tests final-digit label corruption rather than every possible type of incorrect answer. I evaluated both models on the same 1,000 clean examples, using the same initialization seed, dimensions, batch size, Adam settings, clipping, and 10-epoch budget.

| Training labels | Best generation EM |
|---|---:|
| Clean | 9.00% |
| 20% final-digit corruption | 7.80% |

The result shows that this label corruption harmed clean-answer generation under the tested configuration. Incorrect supervision can give conflicting gradients for similar inputs and encourage incorrect digit predictions. Since I used one seed and small models, I cannot establish the size of this effect across other configurations.

| Expression | Correct answer | Actual clean-model generation | Actual corrupted-label-model generation |
|---|---|---|---|
| `(1+13)-38=` | `-24` | `-24` | `-21` |
| `9-(6+38)=` | `-35` | `-39` | `-31` |
| `31-(15*15)=` | `-194` | `-233` | `-196` |
| `4-47-21=` | `-64` | `-66` | `-61` |
| `17+(41*41)=` | `1698` | `1661` | `1666` |
| `(32*22)-12=` | `692` | `766` | `796` |

##### 5. Why do we need gradient clipping during training? (5%)

Answer: During backpropagation through time, RNNs and LSTMs repeatedly multiply Jacobians across time steps. This can cause exploding gradients, excessively large parameter updates, and even nonfinite losses. I used clip_grad_value_(parameters, 1) after loss.backward() and before optimizer.step() to restrict each gradient element to [-1,1]. This is elementwise clipping rather than clipping the overall gradient norm. It does not solve vanishing gradients or guarantee convergence. I did not run a separate clipping ablation, so this explanation describes its theoretical purpose rather than a measured comparison.

##### … Anything that can strengthen your report. (5%)

Answer: I checked the original 1,000,000 training examples and 10,000 evaluation examples: neither set contained missing values, and the source-string intersection was empty. I used the evaluation set for model selection without adding it to training. My T4 baseline with embedding/hidden=256/256 achieved EM scores of 49.73% and 55.20% in its first two epochs. The following table contains measured results from my MPS experiment. The baseline and main run differ in model settings, batch size, and device, so this is not an ablation that controls all variables.

| Epoch | Full-answer generation EM | Learning rate | Recorded seconds |
|---|---:|---:|---:|
| 1 | 52.75% | 0.001 | 327.0 |
| 2 | 64.28% | 0.001 | 297.1 |
| 3 | 72.69% | 0.001 | 305.9 |
| 4 | 78.58% | 0.001 | 294.7 |
| 5 | 82.66% | 0.001 | 305.8 |
| 6 | 84.27% | 0.001 | 310.3 |
| 7 | 88.07% | 0.001 | 285.9 |
| 8 | 89.49% | 0.001 | 292.3 |
| 9 | 89.49% | 0.001 | 278.2 |
| 10 | 92.58% | 0.001 | 274.7 |
| 11 | 93.01% | 0.001 | 285.0 |
| 12 | 93.94% | 0.001 | 282.4 |
| 13 | 94.11% | 0.001 | 298.3 |
| 14 | 93.14% | 0.001 | 297.0 |
| 15 | 94.58% | 0.001 | 342.8 |
| 16 | 94.57% | 0.001 | 469.5 |
| 17 | 94.98% | 0.001 | 314.6 |
| 18 | 94.79% | 0.001 | 302.3 |
| 19 | 94.78% | 0.001 | 422.7 |
| 20 | 97.72% | 0.0005 | 662.3 |
| 21 | 97.36% | 0.0005 | 156.5 |
| 22 | 97.72% | 0.0005 | 873.4 |
| 23 | 98.64% | 0.00025 | 832.0 |
| 24 | 98.74% | 0.00025 | 34.3 |
| 25 | 98.70% | 0.00025 | 764.9 |
| 26 | 98.85% | 0.00025 | 871.4 |
| 27 | 98.84% | 0.00025 | 774.8 |
| 28 | 98.79% | 0.00025 | 822.6 |

For an epoch whose evaluation was resumed from an already-trained state, the recorded duration covers the resumed evaluation only, rather than the full training epoch.

I analyzed the selected checkpoint by expression and answer characteristics:

| Group | EM | Correct / total |
|---|---:|---:|
| answer_length_2 | 99.97% | 3397/3398 |
| addition_subtraction | 100.00% | 4178/4178 |
| no_parentheses | 97.83% | 4201/4294 |
| nonnegative_answer | 98.51% | 7210/7319 |
| answer_length_3 | 99.50% | 3788/3807 |
| negative_answer | 99.78% | 2675/2681 |
| answer_length_5 | 88.52% | 324/366 |
| multiplication | 98.02% | 5707/5822 |
| parentheses | 99.61% | 5684/5706 |
| answer_length_4 | 97.29% | 1903/1956 |
| answer_length_1 | 100.00% | 473/473 |

The model achieved 100% EM on addition/subtraction expressions, while multiplication and five-character answers remained more difficult. These are descriptive results for this evaluation set, rather than controlled comparisons of expression difficulty.

I observed the following errors during independent verification:

| Expression | Correct answer | Actual generation |
|---|---:|---:|
| `(35*41)+20=` | `1455` | `1465` |
| `43*40*47=` | `80840` | `79320` |
| `43*39*19=` | `31863` | `31663` |
| `47*14*41=` | `26978` | `27162` |
| `23*21*33=` | `15939` | `16399` |
| `39*43*5=` | `8385` | `8515` |
| `8*(15-46)=` | `-248` | `-264` |
| `(46*48)-7=` | `2201` | `2191` |

I ran training with an experiment launcher while preserving the provided notebook's cell structure and training-loop skeleton. The submitted checkpoint.pt is a plain state_dict that can be loaded with the same CharRNN and vocabulary. For independent CPU verification, I allowed up to 16 generated tokens and compared complete answer strings for each expression. I stored all checkpoint tensors on CPU for portability across CPU, CUDA, and MPS; their values are identical to those of the verified GPU checkpoint.

I also evaluated the selected checkpoint on a synthetic diagnostic set of 10,000 three-operand expressions with operands from 0–49. Its source strings do not overlap with either the training or evaluation set. I used it only for inference diagnostics, without adding it to training. Full-answer EM was 93.92% (9392/10000).

The synthetic set has a different distribution: 9.39% of its expressions contain a literal zero operand, compared with 0.52% of the original evaluation set. I therefore treat it as an additional stress test. This distribution difference alone does not establish the cause of the EM gap.

I analyzed equivalent inputs by reordering expressions using commutativity and associativity, retaining numbers as input symbols without numerical computation or distributive expansion. In the original evaluation set, 91.12% of expressions have a reordered equivalent form in training, compared with 45.19% in the synthetic diagnostic set. Thus, disjoint source strings do not necessarily represent entirely new computation combinations. High EM on the original evaluation set alone does not demonstrate a general arithmetic algorithm. I used this procedure only for dataset analysis, never to generate model answers.


**Generative AI Usage:** Used OpenAI Codex as a coding assistant under my direction. I defined the requirements, selected the approaches, reviewed and tested the generated code, and iteratively refined the implementation.

API references: [PyTorch LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html), [ReduceLROnPlateau](https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.ReduceLROnPlateau.html).
